In [ ]:
import os
from dotenv import load_dotenv

# Loads OPENAI_API_KEY / TAVILY_API_KEY / GOOGLE_API_KEY from the .env file
# in the project root. Never hardcode keys in this notebook.
load_dotenv()


# Adaptive Claim-Level Verified Retrieval Generation (ACVRG)

                ┌───────────────┐
User Query ───► │ Claim Scorer  │
                └───────┬───────┘
                        │
               Score ≥ 70?
                │            │
               YES          NO
                │            │
         Direct Response   Domain Classifier
                                │
                        Domain Router
                                │
                  Trusted Domain Retriever
                                │
                         Answer Generator



In [10]:
pip install langgraph langchain langchain-openai langchain-community faiss-cpu tavily-python



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_community.tools.tavily_search import TavilySearchResults
from typing import TypedDict, Optional, List
import re


In [2]:

gen_llm = ChatOpenAI(model="gpt-4o-mini", 
                     api_key=os.getenv("OPENAI_API_KEY"),
                     temperature=0)
critic_llm = ChatOpenAI(model="gpt-4o-mini", 
                     api_key=os.getenv("OPENAI_API_KEY"),
                     temperature=0)
rag_llm = ChatOpenAI(model="gpt-4o-mini", 
                     api_key=os.getenv("OPENAI_API_KEY"),
                     temperature=0)

In [63]:
class GraphState(TypedDict):
    query: str
    answer: str
    claims: List[str]
    domain: Optional[str]
    retrieved_docs: Optional[List]
    score: float
    iteration: int
    trusted_sources_used:Optional[List]
    
# class GraphState(TypedDict):
#     query: str
#     score: float
#     domain: Optional[str]
#     retrieved_docs: Optional[List]
#     final_answer: Optional[str]


can also add local database

In [14]:
# from langchain_openai import OpenAIEmbeddings
# from langchain_community.vectorstores import FAISS
# from langchain_core.documents import Document

# emb = OpenAIEmbeddings (api_key=os.getenv("OPENAI_API_KEY"))


# docs = [
#     Document(page_content="Insulin regulates blood sugar by enabling glucose uptake."),
#     Document(page_content="Australia's capital is Canberra."),
#     Document(page_content="The liver stores glucose as glycogen.")
# ]

# vectorstore = FAISS.from_documents(docs, emb)

# vectorstore.save_local("db")
# vectorstore = FAISS.load_local("db", emb, allow_dangerous_deserialization=True)
# retriever = vectorstore.as_retriever()



In [6]:
gen_prompt = ChatPromptTemplate.from_template(
"Answer the query:\n{query}"
)
generation_chain = gen_prompt | gen_llm | StrOutputParser()


In [7]:
def generate(state):
    ans = generation_chain.invoke({"query": state["query"]})
    return {"answer": ans, "iteration": 0}

In [8]:
claim_prompt = ChatPromptTemplate.from_template(
"""
Break the answer into atomic factual claims.
Answer:
{answer}
"""
)
claim_chain = claim_prompt | critic_llm | StrOutputParser()


In [9]:
# also add question to find the releatedness of claim to the question
# verify_prompt = ChatPromptTemplate.from_template(
# """
# Score factual correctness of the claim using context.

# Context:
# {context}

# Claim:
# {claim}

# Return ONLY a number between 0 and 100.
# Do not write anything else.
# """
# )
verify_prompt = ChatPromptTemplate.from_template("""
You are a strict factual verifier.

Context:
{context}

Claim:
{claim}

Classify the claim as exactly one of:
SUPPORTED
CONTRADICTED
NOT_FOUND

Return ONLY one word.
""")

LABEL_SCORES = {
    "SUPPORTED": 1.0,
    "CONTRADICTED": 0.0,
    "NOT_FOUND": 0.5
}

def parse_label(text):
    text = text.strip().upper()

    if "SUPPORTED" in text:
        return "SUPPORTED"
    if "CONTRADICTED" in text:
        return "CONTRADICTED"
    if "NOT_FOUND" in text:
        return "NOT_FOUND"

    return "NOT_FOUND"

verify_chain = verify_prompt | critic_llm | StrOutputParser()


In [89]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

llm_gemnni = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.5
)

In [97]:
verify_prompt_only_LLM = ChatPromptTemplate.from_template("""
You are a strict factual verifier, analyze the claim and query

query:
{query}
                                                          

Claim:
{claim}

Classify the claim as exactly one of:
SUPPORTED (if the claim is factually correct and related to the query)
CONTRADICTED (if claim is factually incorrect )
NOT_FOUND (other)

Return ONLY one word.
""")

LABEL_SCORES = {
    "SUPPORTED": 1.0,
    "CONTRADICTED": 0.0,
    "NOT_FOUND": 0.5
}


verify_chain_llm = verify_prompt_only_LLM | llm_gemnni | StrOutputParser()
def score_claims_pairwise_only_LLM(state):

    claims = state["claims"]
    # retrieved_docs = state["retrieved_docs"]
    query=state["query"]
    results = []

    for claim in claims:

        res = verify_chain_llm.invoke({
            "claim": claim,
            # "context": retrieved_docs
            "query":query
        })

        label = parse_label(res)
        score = LABEL_SCORES[label]

        results.append(score)

    final_score = sum(results) / len(results) if results else 0

    return {
        "score": final_score * 100,
        "labels": results
    }


In [10]:
TRUSTED_SOURCES = {
    "law": [
        "indiankanoon.org",
        "supremecourtofindia.nic.in",
        "legislative.gov.in"
    ],
    "engineering": [
        "ieeexplore.ieee.org",
        "asme.org",
        "sciencedirect.com"
    ],
    "history": [
        "britannica.com",
        "nationalarchives.gov",
        "history.com"
    ],
    "finance": [
        "rbi.org.in",
        "imf.org",
        "worldbank.org",
        "investopedia.com"
    ],
    "medical": [
        "pubmed.ncbi.nlm.nih.gov",
        "who.int",
        "mayoclinic.org",
        "nih.gov"
    ],
    "general": [
        "britannica.com",
        "wikipedia.org"
    ]
}


In [11]:
def detect_domain(state):
    query = state["query"]

    prompt = f"""
Classify the domain of this query into ONE of:
law, engineering, history, finance, medical, general.

Query: {query}

Return only the domain name.
"""

    response = gen_llm.invoke(prompt)
    domain = response.content.strip().lower()

    if domain not in TRUSTED_SOURCES:
        domain = "general"

    return {"domain": domain}


In [68]:

from langchain_community.tools import TavilySearchResults


def trusted_search(state):

    question = state["query"]
    domain = state["domain"]

    allowed_domains = TRUSTED_SOURCES.get(domain, TRUSTED_SOURCES["general"])

    # Create strict site-filtered query
    site_filter = " OR ".join([f"site:{d}" for d in allowed_domains])
    strict_query = f"{site_filter} {question}"

    search = TavilySearchResults(
        tavily_api_key=os.getenv("TAVILY_API_KEY"),
        k=5
    )

    results = search.invoke({
        "query": strict_query
    })

    # Extract clean text
    context = "\n".join([r["content"] for r in results if "content" in r])

    return {
        "retrieved_docs": context,
        "trusted_sources_used": allowed_domains
    }


In [13]:
def extract_score(text):
    try:
        num = float(text.strip())
        return max(0, min(num, 100))
    except:
        import re
        m = re.search(r'\b(\d{1,3})\b', text)
        return float(m.group(1)) if m else 0



In [14]:
def extract_claims(state):
    cl = claim_chain.invoke({"answer": state["answer"]})
    claims = [c for c in cl.split("\n") if c.strip()]
    return {"claims": claims}


In [25]:
def retrieve(state):
    # vec_docs = retriever.invoke(state["question"])
    web_tool = TavilySearchResults(tavily_api_key=os.getenv("TAVILY_API_KEY"),k=3)

    web_docs = web_tool.invoke(state["query"])

    # context += "\n".join([d.page_content for d in vec_docs])
    # context = "\n".join([w["content"] for w in web_docs])

    return {"retrieved_docs": web_docs}


In [16]:
# def score_claims(state):
#     scores = []

#     for claim in state["claims"]:
#         res = verify_chain.invoke({
#             "claim": claim,
#             "context": state["context"]
#         })
#         scores.append(extract_score(res))

#     score = sum(scores)/len(scores) if scores else 0
#     return {"score": score}
def score_claims_pairwise(state):

    claims = state["claims"]
    retrieved_docs = state["retrieved_docs"]

    results = []

    for claim in claims:

        res = verify_chain.invoke({
            "claim": claim,
            "context": retrieved_docs
        })

        label = parse_label(res)
        score = LABEL_SCORES[label]

        results.append(score)

    final_score = sum(results) / len(results) if results else 0

    return {
        "score": final_score * 100,
        "labels": results
    }


# Even Stronger (Optional Upgrade)

# Add contradiction penalty:

# if any(label == "CONTRADICTED"):
#     final_score *= 0.7

# This penalizes harmful hallucinations more strongly.

Important: Experimental Metric Section

Now you can report:

% Supported claims

% Contradicted claims

% Not Found claims

Hallucination Rate = Contradicted / Total

Unsupported Rate = Not Found / Total

Reviewers love these breakdowns.

In [23]:
# if context is not related to questing like amu que ,first context regected by regenerate node like amu ques and then it get less score, finally after 3 itr it get very loc score
def regenerate(state):
    prompt = ChatPromptTemplate.from_template(
    """
    Use context to answer:

    Context:
    {context}

    Question:
    {question}
    """
    )

    chain = prompt | rag_llm | StrOutputParser()

    ans = chain.invoke({
        "context": state["retrieved_docs"],
        "question": state["query"]
    })

    return {
        "answer": ans,
        "iteration": state["iteration"] + 1
    }


In [18]:
# def check_score(state):
#     if state["score"] >= 70:
#         return "good"
#     if state["iteration"] >= 3:
#         return "stop"
#     return "retry"

def route_on_score(state):
    if state["score"] >= 70:
        return "generate_answer"
    if state["iteration"] >= 3:
        return "stop"
    else:
        return "detect_domain"


In [114]:
critic_prompt = ChatPromptTemplate.from_template(
"""
"On a scale of 0–100, how confident are you this answer is factually correct?"


query:
{query}

Answer:
{answer}

Return only score.
"""
)
critic_chain = critic_prompt | critic_llm | StrOutputParser()


In [115]:
def critic(state):
    res = critic_chain.invoke({
        "query": state["query"],
        "answer": state["answer"]
    })
    return {"score": extract_score(res)}

### now add what sir wants , domain specific retrieval, domain ddetection , web search sould be in authentic website . and can also add local database , domain routing : after domain detection do routing

In [116]:
builder = StateGraph(GraphState)

builder.add_node("generate", generate)
builder.add_node("extract_claims", extract_claims)
builder.add_node("critic", critic)
builder.add_node("detect_domain", detect_domain)
builder.add_node("retrieve", retrieve)
builder.add_node("trusted_search", trusted_search)
builder.add_node("score_claims_pairwise", score_claims_pairwise)
builder.add_node("regenerate", regenerate)

builder.set_entry_point("generate")

builder.add_edge("generate", "critic")
# builder.add_edge("critic", "retrieve")
# builder.add_edge("retrieve", "score_claims_pairwise")

builder.add_conditional_edges(
    "critic",
    route_on_score,
    {
        "generate_answer": END,
        "detect_domain": "detect_domain",
        "stop": END
    }
)

builder.add_edge("detect_domain", "trusted_search")
builder.add_edge("trusted_search", "regenerate")
builder.add_edge("regenerate", "critic")
# builder.add_edge("extract_claims", "score_claims_pairwise")




graph = builder.compile()

In [ ]:
# builder = StateGraph(GraphState)

# builder.add_node("generate", generate)
# builder.add_node("extract_claims", extract_claims)
# builder.add_node("detect_domain", detect_domain)
# builder.add_node("retrieve", retrieve)
# builder.add_node("trusted_search", trusted_search)
# builder.add_node("score_claims_pairwise", score_claims_pairwise)
# builder.add_node("regenerate", regenerate)

# builder.set_entry_point("generate")

# builder.add_edge("generate", "extract_claims")
# builder.add_edge("extract_claims", "retrieve")
# builder.add_edge("retrieve", "score_claims_pairwise")

# builder.add_conditional_edges(
#     "score_claims_pairwise",
#     route_on_score,
#     {
#         "generate_answer": END,
#         "detect_domain": "detect_domain",
#         "stop": END
#     }
# )

# builder.add_edge("detect_domain", "trusted_search")
# builder.add_edge("trusted_search", "regenerate")
# builder.add_edge("regenerate", "extract_claims")





# graph = builder.compile()





In [ ]:
# builder = StateGraph(GraphState)

# builder.add_node("generate", generate)
# builder.add_node("extract_claims", extract_claims)
# builder.add_node("detect_domain", detect_domain)
# builder.add_node("retrieve", retrieve)
# builder.add_node("trusted_search", trusted_search)
# builder.add_node("score_claims_pairwise", score_claims_pairwise)
# builder.add_node("score_claims_pairwise_only_LLM", score_claims_pairwise_only_LLM)

# builder.add_node("regenerate", regenerate)

# builder.set_entry_point("generate")

# builder.add_edge("generate", "extract_claims")
# builder.add_edge("extract_claims", "score_claims_pairwise_only_LLM")
# # builder.add_edge("retrieve", "score_claims_pairwise")

# builder.add_conditional_edges(
#     "score_claims_pairwise_only_LLM",
#     route_on_score,
#     {
#         "generate_answer": END,
#         "detect_domain": "detect_domain",
#         "stop": END
#     }
# )

# builder.add_edge("detect_domain", "trusted_search")
# builder.add_edge("trusted_search", "regenerate")
# builder.add_edge("regenerate", "extract_claims")
# builder.add_edge("extract_claims", "score_claims_pairwise_only_LLM")




# graph = builder.compile()


In [ ]:
result = graph.invoke({
    "question": "Explain how insulin regulates blood sugar"
})

print(result)


In [ ]:
inputs = {"query": "Explain how insulin regulates blood sugar"}


In [52]:
for step in graph.stream(inputs):
    print(step)

{'generate': {'answer': 'Insulin is a hormone produced by the pancreas that plays a crucial role in regulating blood sugar (glucose) levels in the body. Here’s how it works:\n\n1. **Glucose Uptake**: After you eat, carbohydrates in food are broken down into glucose, which enters the bloodstream. This increase in blood glucose levels signals the pancreas to release insulin.\n\n2. **Facilitating Cellular Uptake**: Insulin facilitates the uptake of glucose by cells, particularly in muscle and fat tissues. It does this by binding to insulin receptors on the cell surface, which triggers a series of cellular processes that allow glucose transporters (such as GLUT4) to move to the cell membrane. These transporters then help glucose enter the cells.\n\n3. **Storage of Glucose**: In addition to promoting glucose uptake, insulin encourages the liver and muscle cells to store excess glucose in the form of glycogen (a process called glycogenesis). This helps lower blood sugar levels after meals.\n

In [117]:
for step in graph.stream({"query": "Give 5 research papers (with DOI) about quantum blockchain"}):
    print(step)

{'generate': {'answer': 'Here are five research papers related to quantum blockchain, along with their DOIs:\n\n1. **Title:** Quantum Blockchain: A Survey\n   - **Authors:** Z. Zhang, Y. Zhang, Y. Liu, and H. Wang\n   - **Journal:** IEEE Access\n   - **Year:** 2020\n   - **DOI:** [10.1109/ACCESS.2020.2991234](https://doi.org/10.1109/ACCESS.2020.2991234)\n\n2. **Title:** Quantum Blockchain: A New Paradigm for Secure Transactions\n   - **Authors:** M. A. A. Alzahrani, A. A. Alhassan, and A. A. Alzahrani\n   - **Journal:** IEEE Transactions on Quantum Engineering\n   - **Year:** 2021\n   - **DOI:** [10.1109/TQE.2021.3071234](https://doi.org/10.1109/TQE.2021.3071234)\n\n3. **Title:** Quantum-Resistant Blockchain Protocols\n   - **Authors:** A. M. A. Alhassan, M. A. A. Alzahrani, and A. A. Alzahrani\n   - **Journal:** Future Generation Computer Systems\n   - **Year:** 2022\n   - **DOI:** [10.1016/j.future.2021.10.012](https://doi.org/10.1016/j.future.2021.10.012)\n\n4. **Title:** A Quantum 

In [118]:
for step in graph.stream({"query": "List Supreme Court cases from 1890"}):
    print(step)

{'generate': {'answer': "In 1890, several notable Supreme Court cases were decided. Here are a few significant ones:\n\n1. **McCulloch v. Maryland (1819)** - Although this case is from an earlier date, it was often referenced in discussions about federal power and state rights during the 1890s.\n\n2. **United States v. E.C. Knight Co. (1895)** - This case involved the Sherman Antitrust Act and addressed the limits of federal power in regulating monopolies.\n\n3. **In re Debs (1895)** - This case involved labor strikes and the federal government's authority to intervene in labor disputes, particularly in the context of the Pullman Strike.\n\n4. **Chicago, Milwaukee & St. Paul Railway Co. v. Minnesota (1890)** - This case dealt with the regulation of railroad rates and the authority of state commissions.\n\n5. **Nix v. Hedden (1893)** - Although decided a few years later, it is often discussed in the context of tariff laws and the classification of goods.\n\nWhile the list includes cases

In [ ]:
for step in graph.stream({"query": "List Supreme Court cases from 1890"}):
    print(step)

{'generate': {'answer': "In 1890, there were several notable Supreme Court cases. Here are a few significant ones:\n\n1. **McCulloch v. Maryland (1819)** - Although this case is from an earlier date, it was often referenced in legal discussions during the 1890s regarding federal power and state rights.\n\n2. **United States v. E.C. Knight Co. (1895)** - This case, while technically decided in 1895, stemmed from issues that were prominent in the late 1880s and early 1890s regarding the Sherman Antitrust Act and the regulation of monopolies.\n\n3. **In re Debs (1895)** - This case involved the federal government's authority to intervene in labor disputes, particularly during the Pullman Strike, and while it was decided in 1895, it was rooted in the labor issues of the time.\n\n4. **Chicago, Milwaukee & St. Paul Railway Co. v. Minnesota (1890)** - This case addressed the issue of state regulation of railroad rates and the due process rights of the railroads.\n\n5. **Nix v. Hedden (1893)**

In [44]:
for step in graph.stream({"question": "List Supreme Court of india cases from 1890"}):
    print(step)

{'generate': {'answer': 'The Supreme Court of India was established in 1950, following the adoption of the Constitution of India. However, prior to that, the judicial system in India was governed by various colonial courts, including the Privy Council and High Courts established under British rule.\n\nIf you are looking for significant cases or legal developments from the late 19th century that influenced the Indian legal system, you might consider cases from the Privy Council or the High Courts established during that time. Some notable cases from the late 19th century include:\n\n1. **Queen Empress v. Khairati** (1880) - This case dealt with issues of criminal law and the interpretation of statutes.\n2. **Mohan Lal v. State of U.P.** (1890) - This case involved property rights and the interpretation of land revenue laws.\n\nIf you are specifically interested in cases that would later influence the Supreme Court of India or its jurisprudence, please clarify, and I can provide more tar

In [43]:
for step in graph.stream({"question": "List 20 historically significant events that happened in Aligarh Muslim University between the vears 1750 to 1850"}):
    print(step)

{'generate': {'answer': "Aligarh Muslim University (AMU) was established in 1920, but its roots can be traced back to the Muhammadan Anglo-Oriental College (MAO College), which was founded in 1875. Therefore, there are no events specifically related to Aligarh Muslim University between 1750 and 1850, as the institution itself did not exist during that time. However, I can provide a list of significant events related to the broader context of education, social reform, and the Muslim community in India during that period, which eventually contributed to the establishment of AMU. Here are some notable events:\n\n1. **1757** - The Battle of Plassey, leading to British control over Bengal and significant changes in Indian society.\n2. **1780** - Establishment of the Fort William College in Calcutta, promoting English education.\n3. **1790** - The founding of the Calcutta Madrasah, one of the first institutions for Muslim education in India.\n4. **1820** - The establishment of the first Engl